# Conversation Memory and the Multi-Turn Session Loop

**Tutorial reference:** Part 6 — Layer 5 of the SmartIntake Learning Tutorial

This notebook covers the final layer: maintaining conversation state across turns and routing each session through the correct path (happy, partial, or fallback).

The key rule: **once a field is provided by the user, it must never be asked for again.**

---

## 1. Setup

In [1]:
from dotenv import load_dotenv
import os
import json
from datetime import datetime
from pydantic import BaseModel, field_validator, ValidationError
from typing import Literal, Optional
from langchain_anthropic import ChatAnthropic
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.memory import ConversationBufferMemory, ConversationSummaryMemory

load_dotenv()

llm = ChatAnthropic(
    model="claude-sonnet-4-6",
    anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
    max_tokens=512
)

os.makedirs("output", exist_ok=True)
print("Setup complete.")


Setup complete.


## 2. The IntakeRecord Model

Copied from NB-04 — the canonical Pydantic model for the project.

In [2]:
class IntakeRecord(BaseModel):
    query_type: Literal[
        "complaint", "submission", "variation", "safety_signal",
        "label_update", "inspection", "general_enquiry"
    ]
    regulation_ref: Literal[
        "FDA_21CFR", "EMA_CTR", "ICH_E2A", "ICH_Q10",
        "CDSCO_NDC", "GxP_GMP", "GxP_GCP", "other"
    ]
    product_area: Literal[
        "oncology", "cardiovascular", "infectious_disease",
        "cmc", "clinical", "labelling", "general"
    ]
    urgency: Literal["routine", "standard", "urgent", "critical"]
    submitting_team: str

    @field_validator("submitting_team")
    @classmethod
    def team_must_not_be_empty(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("submitting_team cannot be empty")
        return v.strip().title()

    @field_validator("submitting_team")
    @classmethod
    def team_not_a_person(cls, v: str) -> str:
        tokens = v.strip().split()
        if len(tokens) == 1 and tokens[0][0].isupper() and tokens[0].isalpha():
            raise ValueError(
                "submitting_team appears to be an individual name. "
                "Please provide a team or function name."
            )
        return v


print("IntakeRecord model ready.")

IntakeRecord model ready.


## 3. ConversationBufferMemory

`ConversationBufferMemory` stores every message verbatim. It is the right choice for short sessions (up to 10 turns) where full fidelity matters more than token efficiency.

In [ ]:
# Demonstrate how ConversationBufferMemory accumulates messages.
# In the real session loop, we inject the memory into each API call as context.

buffer_memory = ConversationBufferMemory(return_messages=True)

# Simulate adding turns to the memory
buffer_memory.chat_memory.add_user_message(
    "Hi, we need to file a Type II variation for our cardiovascular product with the EMA."
)
buffer_memory.chat_memory.add_ai_message(
    "I have noted the variation type and regulation. Could you tell me the urgency level "
    "— is there a specific regulatory deadline for this filing?"
)
buffer_memory.chat_memory.add_user_message("The deadline is in about 3 months — standard priority.")

# Load the messages — this is what we inject into the next API call
history = buffer_memory.load_memory_variables({})
print("Buffer memory contents after 3 turns:")
for msg in history["history"]:
    role = msg.__class__.__name__.replace("Message", "")
    print(f"  [{role}] {msg.content[:80]}")

## 4. ConversationSummaryMemory

`ConversationSummaryMemory` compresses the conversation into a running summary after each turn. Use this when sessions exceed 10 turns to keep token usage manageable.

In [ ]:
# ConversationSummaryMemory uses the LLM itself to generate the summary.
# It requires the llm parameter at construction time.

summary_memory = ConversationSummaryMemory(llm=llm, return_messages=False)

# Add the same three turns as above
summary_memory.save_context(
    {"input": "Hi, we need to file a Type II variation for our cardiovascular product with the EMA."},
    {"output": "I have noted the variation type. Could you confirm the urgency and submitting team?"}
)
summary_memory.save_context(
    {"input": "The deadline is in about 3 months — standard priority."},
    {"output": "Thank you. Could you confirm which team is submitting this query?"}
)

summary = summary_memory.load_memory_variables({})
print("Summary memory after 2 exchanges:")
print(summary["history"])

## 5. The Memory Switching Logic

Use buffer memory for sessions up to 10 turns; switch to summary memory beyond that.

In [3]:
# Memory manager that switches between buffer and summary based on turn count.
# In a real implementation this would transfer the existing history on switch.

BUFFER_LIMIT = 10

class MemoryManager:
    """
    Manages conversation memory with automatic switching at BUFFER_LIMIT turns.
    
    Uses ConversationBufferMemory for short sessions (full fidelity).
    Switches to ConversationSummaryMemory for long sessions (token efficiency).
    """

    def __init__(self, llm):
        self.llm = llm
        self.turn_count = 0
        self._memory = ConversationBufferMemory(return_messages=True)
        self._using_summary = False

    def add_turn(self, user_message: str, assistant_response: str):
        """Record a completed turn and switch memory type if the limit is reached."""
        self.turn_count += 1

        # Switch to summary memory after BUFFER_LIMIT turns
        if self.turn_count == BUFFER_LIMIT and not self._using_summary:
            print(f"[Memory] Switching to ConversationSummaryMemory at turn {self.turn_count}.")
            self._memory = ConversationSummaryMemory(llm=self.llm, return_messages=False)
            self._using_summary = True

        if self._using_summary:
            self._memory.save_context(
                {"input": user_message},
                {"output": assistant_response}
            )
        else:
            self._memory.chat_memory.add_user_message(user_message)
            self._memory.chat_memory.add_ai_message(assistant_response)

    def get_history_text(self) -> str:
        """Return the conversation history as a plain text string for prompt injection."""
        history = self._memory.load_memory_variables({})
        if self._using_summary:
            return history.get("history", "")
        else:
            messages = history.get("history", [])
            lines = []
            for msg in messages:
                role = "User" if msg.__class__.__name__ == "HumanMessage" else "Assistant"
                lines.append(f"{role}: {msg.content}")
            return "\n".join(lines)


# Quick test
mem = MemoryManager(llm)
mem.add_turn(
    "We need to file a variation for our cardiovascular product.",
    "Noted. Could you confirm the urgency level?"
)
mem.add_turn(
    "Standard priority — deadline in 3 months.",
    "Thank you. Which team is submitting this?"
)
print("History text:")
print(mem.get_history_text())

History text:
User: We need to file a variation for our cardiovascular product.
Assistant: Noted. Could you confirm the urgency level?
User: Standard priority — deadline in 3 months.
Assistant: Thank you. Which team is submitting this?


/var/folders/_g/x5x8p4910pzdcn9kd8gl4lrm0000gn/T/ipykernel_28500/2588372056.py:17: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  self._memory = ConversationBufferMemory(return_messages=True)


## 6. The Core Extraction Logic With Memory

The key design: pass the `collected_fields` dict to the model in every extraction call. The model is told which fields are already confirmed and asked only for what is still missing.

In [4]:
MULTI_TURN_EXTRACTION_SYSTEM = """
You are SmartIntake, a regulatory affairs intake specialist.
You are in an ongoing conversation collecting five regulatory intake fields.

Fields to collect:
  query_type: complaint | submission | variation | safety_signal | label_update | inspection | general_enquiry
  regulation_ref: FDA_21CFR | EMA_CTR | ICH_E2A | ICH_Q10 | CDSCO_NDC | GxP_GMP | GxP_GCP | other
  product_area: oncology | cardiovascular | infectious_disease | cmc | clinical | labelling | general
  urgency: routine | standard | urgent | critical
  submitting_team: team or function name — never an individual person's name

RULES:
- NEVER ask for a field that is already in 'Already collected fields'.
- Extract any new field values from the latest user message.
- NEVER infer urgency from tone — only from an explicit deadline.
- If the input is not a regulatory query, respond with: {{"error": "non_regulatory_input"}}

Return ONLY a JSON object with the fields you can extract from the latest message.
Do not repeat already-collected fields. Do not invent values.
"""

multi_turn_prompt = ChatPromptTemplate.from_messages([
    ("system", MULTI_TURN_EXTRACTION_SYSTEM),
    ("human",
     "Conversation history:\n{history}\n\n"
     "Already collected fields: {collected_fields}\n\n"
     "Latest message: {user_message}")
])

multi_turn_chain = multi_turn_prompt | llm | StrOutputParser()


# def parse_json_output(text: str) -> dict:
#     """Strip markdown fences and parse JSON."""
#     clean = (
#         text.strip()
#         .removeprefix("```json").removeprefix("```")
#         .removesuffix("```")
#         .strip()
#     )
#     return json.loads(clean)

def parse_json_output(text: str) -> dict:
    """
    Strip markdown code fences and parse the JSON from the LLM's text output.
    
    The model sometimes wraps JSON in ```json ... ``` fences even when instructed
    not to. This function handles both fenced and unfenced output.
    """
    clean = (
        text.strip()
        .removeprefix("```json")
        .removeprefix("```")
        .removesuffix("```")
        .strip()
    )
    return json.loads(clean)


print("Multi-turn extraction chain defined.")

Multi-turn extraction chain defined.


## 7. Simulating the Partial Path (T2)

> With Memory

T2 provides `query_type`, `regulation_ref`, and `product_area` but not `urgency` or `submitting_team`. We simulate three turns and verify that already-collected fields are never re-requested.

In [5]:
# ===== DEMO: Correction scenario =====
# WITH memory: Claude knows turn 2 overrides turn 1
# WITHOUT memory: Claude has no idea what "Wait, I made a mistake" refers to

simulated_turns = [
    "PV team here. This is a safety signal for our clinical trial per ICH E2A.",
    "Wait, actually I was wrong. This is a CMC submission for a manufacturing site, not a safety signal.",
]

print("=" * 70)
print("WITH MEMORY - Claude sees the full conversation:")
print("=" * 70)
mem = MemoryManager(llm)
collected = {}
for msg in simulated_turns:
    raw = multi_turn_chain.invoke({
        "history": mem.get_history_text(),  # <-- Knows turn 1 context
        "collected_fields": json.dumps(collected),
        "user_message": msg
    })
    new = parse_json_output(raw)
    for k, v in new.items():
        if v is not None:
            collected[k] = v
    follow_up = f"Ok. Collected: {list(collected.keys())}"
    mem.add_turn(msg, follow_up)
print(f"query_type: {collected.get('query_type')}")    # "submission" ✓ (corrected from "safety_signal")
print(f"regulation_ref: {collected.get('regulation_ref')}")  # "FDA_21CFR" ✓ (corrected from "ICH_E2A")
print(f"product_area: {collected.get('product_area')}")     # "cmc" ✓ (corrected from "clinical")
print(f"ALL: {json.dumps(collected, indent=2)}")
# Claude understands: "actually I was wrong" refers back to the first message
# and replaces those values.

print("\n" + "=" * 70)
print("WITHOUT MEMORY - Claude sees each message in isolation:")
print("=" * 70)
collected = {}
for msg in simulated_turns:
    raw = multi_turn_chain.invoke({
        "history": "",  # <-- No context at all
        "collected_fields": json.dumps(collected),
        "user_message": msg
    })
    new = parse_json_output(raw)
    for k, v in new.items():
        if v is not None:
            collected[k] = v
print(f"query_type: {collected.get('query_type')}")      # May still be "safety_signal"
print(f"regulation_ref: {collected.get('regulation_ref')}")  # May still be "ICH_E2A"
print(f"product_area: {collected.get('product_area')}")     # May end up with conflicting values
print(f"ALL: {json.dumps(collected, indent=2)}")
# PROBLEM: Claude sees "Wait, I was wrong..." with NO context of what was wrong.
# The word "manufacturing site" doesn't tell Claude to override "safety_signal" to "submission"
# or "clinical" to "cmc". It has no prior message to anchor the correction to.


WITH MEMORY - Claude sees the full conversation:
query_type: submission
regulation_ref: ICH_E2A
product_area: cmc
ALL: {
  "query_type": "submission",
  "regulation_ref": "ICH_E2A",
  "product_area": "cmc",
  "submitting_team": "PV team"
}

WITHOUT MEMORY - Claude sees each message in isolation:
query_type: submission
regulation_ref: ICH_E2A
product_area: cmc
ALL: {
  "query_type": "submission",
  "regulation_ref": "ICH_E2A",
  "product_area": "cmc",
  "submitting_team": "PV team"
}


In [6]:
# ===== DEMO: Implicit answer =====
# WITH memory: Claude knows "Yes, standard" answers the urgency question
# WITHOUT memory: Claude only sees "Yes, standard" in isolation

simulated_turns = [
    "Labelling team. Need to update the SmPC for our oncology product per EMA.",
    "Yes, standard timeline is fine.",
]

print("=" * 70)
print("WITH MEMORY - Claude remembers the assistant asked about urgency:")
print("=" * 70)
mem = MemoryManager(llm)
collected = {}
for msg in simulated_turns:
    raw = multi_turn_chain.invoke({
        "history": mem.get_history_text(),  # <-- Remembers assistant's follow-up question
        "collected_fields": json.dumps(collected),
        "user_message": msg
    })
    new = parse_json_output(raw)
    for k, v in new.items():
        if v is not None:
            collected[k] = v
    follow_up = f"Compiled: {list(collected.keys())}"
    mem.add_turn(msg, follow_up)
print(f"urgency: {collected.get('urgency')}")  # "standard" ✓ — knows "Yes" refers to the deadline question

print("\n" + "=" * 70)
print("WITHOUT MEMORY - Claude doesn't know what 'Yes' refers to:")
print("=" * 70)
collected = {}
for msg in simulated_turns:
    raw = multi_turn_chain.invoke({
        "history": "",  # <-- No context of the conversation
        "collected_fields": json.dumps(collected),
        "user_message": msg
    })
    new = parse_json_output(raw)
    for k, v in new.items():
        if v is not None:
            collected[k] = v
print(f"urgency: {collected.get('urgency')}")  # May be None — "Yes, standard timeline" has no
                                                  # explicit mention of urgency without context


WITH MEMORY - Claude remembers the assistant asked about urgency:
urgency: standard

WITHOUT MEMORY - Claude doesn't know what 'Yes' refers to:
urgency: standard
